# Phase Cross-Spectral Biosecurity — Exploration Notebook

This notebook walks through the fingerprinting method interactively.
Set `DATA_CSV` to your prepared dataset path before running.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from src.encoding import voss_encode
from src.fingerprint import phase_cross_spectral_fingerprint, fingerprint_dim
from src.baseline import voss_power_fingerprint

DATA_CSV = '../data/sequences.csv'  # set your path

## 1. Voss encoding of a short example sequence

In [ ]:
seq = 'ATGCATGCATGC'
enc = voss_encode(seq)
fig, axes = plt.subplots(4, 1, figsize=(10, 4), sharex=True)
for i, (ax, ch) in enumerate(zip(axes, ['A','T','G','C'])):
    ax.step(range(len(seq)), enc[i], where='mid', label=ch)
    ax.set_ylabel(ch); ax.set_ylim(-0.1, 1.1)
axes[-1].set_xlabel('Position')
fig.suptitle('Voss 4-channel encoding')
plt.tight_layout()

## 2. Feature dimension at W=90 and W=420

In [ ]:
for W in [90, 420]:
    print(f'W={W}: phase dim = {fingerprint_dim(W)}, Voss power dim = {4*(W//2+1)}')

## 3. Compute fingerprints for a pair of sequences

In [ ]:
import pandas as pd
df = pd.read_csv(DATA_CSV)
dangerous = df[df['label'] == 1]['sequence'].iloc[0]
benign    = df[df['label'] == 0]['sequence'].iloc[0]

W = 90
fp_d = phase_cross_spectral_fingerprint(dangerous, W)
fp_b = phase_cross_spectral_fingerprint(benign, W)
print(f'Dangerous FP shape: {fp_d.shape}, Benign FP shape: {fp_b.shape}')

from scipy.spatial.distance import cosine
print(f'Cosine distance (dangerous vs benign): {cosine(fp_d, fp_b):.4f}')

## 4. Run cluster-aware evaluation (small test — 5 seeds)

In [ ]:
from src.evaluate import cluster_aware_eval, bootstrap_ci
from src.utils import load_dataset
from concurrent.futures import ProcessPoolExecutor

accs, seqs, labels, cluster_ids, _ = load_dataset(DATA_CSV)

W = 90
fps_list = [phase_cross_spectral_fingerprint(s, W) for s in seqs]
valid    = [(a, fp, lab, cid) for a, fp, lab, cid
             in zip(accs, fps_list, labels, cluster_ids) if fp is not None]

fps_mat  = np.vstack([v[1] for v in valid])
labs_arr = np.array([v[2] for v in valid])
cids_arr = np.array([v[3] for v in valid])

aucs = cluster_aware_eval(fps_mat, labs_arr, cids_arr, n_seeds=5, pca_components=64)
lo, hi = bootstrap_ci(aucs)
print(f'W=90 Phase (5 seeds): {np.mean(aucs):.4f} ± {np.std(aucs):.4f}  CI=[{lo:.4f},{hi:.4f}]')